# Imports

In [ ]:
# Local application/library specific imports # type: ignore
from pygrex.config import cfg
from pygrex.data_reader import DataReader, GroupInteractionHandler
from pygrex.models.als_model import ALS
from pygrex.recommender.group_recommender import GroupRecommender
from pygrex.utils.association_rules import AssociationRules
from pygrex.explain.rule_based_group_rec_explainer import RuleBasedGroupRecExplainer


In [4]:
# Read the ratings file.
data = DataReader(**cfg.data.test)

print(data.dataset["itemId"].unique() ) # Ensure itemId is unique

data.make_consecutive_ids_in_dataset()

print(data.dataset["itemId"].unique() ) # Ensure itemId is unique

# Train the recommendation model
algo = ALS(**cfg.model.als)
algo.fit(data)

# Read the file with the group ids
group_handler = GroupInteractionHandler(**cfg.data.groups)
all_groups = group_handler.read_groups("groupsWithHighRatings5.txt")

[     1      3      6 ... 160836 163937 163981]
[   0    1    2 ... 9721 9722 9723]


c:\Users\usuar\miniconda3\envs\pygrex-exp-grs\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
association_rules = AssociationRules(data=data, min_support=0.08, min_confidence=0.6, rating_threshold=1) # type: ignore
rules = association_rules.compute() 
df_filtered = association_rules.get_df_filtered_by_rating_threshold()
user_history = df_filtered.groupby("userId")["itemId"].apply(set).to_dict()

In [ ]:
print(user_history) # type: ignore
print(rules) # type: ignore

{0: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 

In [ ]:


for group in all_groups: # type: ignore
    members = group_handler.get_group_members(group) # type: ignore
    print(members)
    print("------------------")
    
    group_recommender = GroupRecommender(data) # type: ignore
    group_recommender.setup_recommendation(algo, members, data) # type: ignore
    original_group_rec = group_recommender.get_group_recommendations(50)
    # get all the items that at least one group member has interacted with
    
    # items_rated_by_group = group_handler.get_rated_items_by_all_group_members(
    #     members, data # type: ignore
    # ) 
    
    explainer = RuleBasedGroupRecExplainer( # type: ignore
        rules=rules, # type: ignore
        data=data, # type: ignore
        pool_recommendations=original_group_rec, # type: ignore
        members=members, # type: ignore
        user_history=user_history, # type: ignore
        min_members_threshold=2
    )
    
    # Find explanations
    explanations = explainer.find_explanation()
    print(f"Group: {group}, Explanations: {explanations}")
    print("------------------")

[522, 385, 234, 452, 594]
------------------
Group: 522_385_234_452_594, Explanations: 0.08
------------------
[522, 385, 234, 246, 428]
------------------
Group: 522_385_234_246_428, Explanations: 0.08
------------------
[452, 246, 220, 586, 82]
------------------
Group: 452_246_220_586_82, Explanations: 0.08
------------------
[452, 246, 220, 586, 198]
------------------
Group: 452_246_220_586_198, Explanations: 0.1
------------------
[452, 246, 220, 586, 50]
------------------
Group: 452_246_220_586_50, Explanations: 0.06
------------------
[220, 586, 73, 263, 372]
------------------
Group: 220_586_73_263_372, Explanations: 0.2
------------------
[220, 586, 73, 263, 365]
------------------
Group: 220_586_73_263_365, Explanations: 0.22
------------------
[220, 586, 73, 263, 6]
------------------
Group: 220_586_73_263_6, Explanations: 0.16
------------------
[73, 263, 563, 119, 66]
------------------
Group: 73_263_563_119_66, Explanations: 0.08
------------------
[73, 263, 563, 4, 312